# Predicting metastasis and metastatic organotropism in malignancies using machine learning

**Ablation notebook** — MSc in Biomedical and Health Informatics, Mahidol University, 2026.

Author: Ching To Chung.

This is the cleaned analysis notebook behind the thesis. It covers:

1. loading the public MSK-IMPACT 2017 cohort (clinical table + MAF),
2. the feature-set ablation — primary site only, mutations only, and combined — with logistic regression and XGBoost,
3. a TabTransformer and an embedding MLP for metastatic-site prediction,
4. binary prediction of primary vs metastatic samples,
5. SHAP explanations for the mutation-based models.

Cell outputs have been cleared so the file stays small and the numbers are reproduced rather than copied. The figures used in the README are in `figures/`.

The dataset is not redistributed here — see `data/README.md` for the source and file names.


## Setup

In [ ]:
# --- Paths -----------------------------------------------------------------
# Put the two MSK-IMPACT files in ./data, or point MSK_DATA_DIR at your download.
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("MSK_DATA_DIR", "data"))
OUTPUT_DIR = Path(os.environ.get("MSK_OUTPUT_DIR", "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLINICAL_TSV = DATA_DIR / "msk_impact_2017_clinical_data.tsv"
MUTATIONS_MAF = DATA_DIR / "data_mutations.txt"

print("clinical :", CLINICAL_TSV, "exists:", CLINICAL_TSV.exists())
print("mutations:", MUTATIONS_MAF, "exists:", MUTATIONS_MAF.exists())

## Importing MSK-IMPACT data (Clinical + Genetic)

In [ ]:
import pandas as pd
df=pd.read_csv(str(CLINICAL_TSV), sep="\t")
df.head()

In [ ]:
df.columns

In [ ]:
import pandas as pd

# === 1. Load the mutation file (MAF format) ===
# Adjust path to where you saved data_mutations.txt
mutations = pd.read_csv(str(MUTATIONS_MAF), sep="\t", comment='#', low_memory=False)

print("File shape:", mutations.shape)
print("Columns:", mutations.columns.tolist())
print(mutations.head())

# === 2. Inspect key fields ===
# Usually, 'Hugo_Symbol' = gene, 'Tumor_Sample_Barcode' = sample ID
print(mutations[['Hugo_Symbol', 'Tumor_Sample_Barcode', 'Variant_Classification']].head())

# === 3. Create a binary mutation matrix ===
# Pivot so each gene is a column, each sample is a row
# Mark 1 if mutated, 0 otherwise
binary_matrix = (
    mutations
    .assign(value=1)
    .pivot_table(index='Tumor_Sample_Barcode',
                 columns='Hugo_Symbol',
                 values='value',
                 aggfunc='max',
                 fill_value=0)
)

print("Binary matrix shape:", binary_matrix.shape)
print(binary_matrix.iloc[0:5, 0:10])  # show first 5 samples × first 10 genes

# === 4. Save the binary matrix ===
binary_matrix.to_csv(OUTPUT_DIR / "mutation_binary_matrix.csv")


In [ ]:
#Merge binary mutation matrix with clinical data
df = df.rename(columns={'Sample ID': 'Tumor_Sample_Barcode'})

merged = df.merge(binary_matrix,
                  how='inner',
                  on='Tumor_Sample_Barcode')

print("Merged shape:", merged.shape)
print("Columns:", merged.columns[:15])  # show first 15 columns

In [ ]:
merged.head()

In [ ]:
merged.to_csv(OUTPUT_DIR / "merged_mutation_clinical.csv", index=False)

## Data Pre-processing for Model Construction

In [ ]:
#Building a logistic regression model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Drop samples without metastatic site annotation
data = merged.dropna(subset=['Metastatic Site']).copy()

# Define y
y = data['Metastatic Site']


print("Number of metastatic cases:", len(y))
print("Number of classes:", y.nunique())
print("Class distribution:\n", y.value_counts())

# Plot a count bar chart for y
plt.figure(figsize=(12, 8))
y.value_counts().plot(kind='bar')
plt.title('Count of Metastatic Sites')
plt.xlabel('Metastatic Site')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Merge rare classes into "Other"
class_counts = y.value_counts()
rare_classes = class_counts[class_counts < 20].index   # threshold can be 2, 5, etc.
data['Metastatic Site'] = data['Metastatic Site'].replace(rare_classes, 'Other')

# Redefine y after replacement
y = data['Metastatic Site']
print("Number of classes:", y.nunique())
print("Class distribution:\n", y.value_counts())
# Plot a count bar chart for y
plt.figure(figsize=(12, 8))
y.value_counts().plot(kind='bar')
plt.title('Count of Metastatic Sites')
plt.xlabel('Metastatic Site')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

In [ ]:
#(1) Primary Site Only
# One-hot encode the categorical "Primary Tumor Site"

primary_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_primary = primary_encoder.fit_transform(data[['Primary Tumor Site']])

# Automatically fetch feature names
primary_feature_names = primary_encoder.get_feature_names_out()
X_primary = pd.DataFrame(X_primary, columns=primary_feature_names, index=data.index)

In [ ]:
#(2) Genetic Mutations Only
# Identify gene columns = all mutation binary features
gene_columns = [col for col in merged.columns if col not in df.columns]  # assuming df is clinical data
X_mut = data[gene_columns].copy()

In [ ]:
#(3) Combined
# Concatenate both primary site and mutation features
X_combined = pd.concat([X_primary, X_mut], axis=1)


## Train / Test Split

In [ ]:
# Use stratified split to preserve distribution of metastatic sites
X_train_p, X_test_p, y_train, y_test = train_test_split(
    X_primary, y, test_size=0.2, random_state=42, stratify=y
)

X_train_m, X_test_m, _, _ = train_test_split(
    X_mut, y, test_size=0.2, random_state=42, stratify=y
)

X_train_c, X_test_c, _, _ = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

## Logistic Model

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

def evaluate_model(clf, X_train, X_test, y_train, y_test, model_name="Model"):
    # Encode labels (because XGBoost requires numeric targets)
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc = le.transform(y_test)

    # Fit on encoded labels
    clf.fit(X_train, y_train_enc)
    y_pred_enc = clf.predict(X_test)

    # Decode predictions back to original string labels
    y_pred = le.inverse_transform(y_pred_enc)

    # Metrics (using original string labels for readability)
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")
    nir = np.max(np.bincount(y_test.factorize()[0])) / len(y_test)

    # Print summary
    print(f"\n=== {model_name} Summary ===")
    print(f"Accuracy: {acc:.3f}")
    print(f"Macro F1: {macro_f1:.3f}")
    print(f"Weighted F1: {weighted_f1:.3f}")
    print(f"No Information Rate: {nir:.3f}")
    print(f"Better than baseline? {'YES' if acc > nir else 'NO'}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    # Confusion matrix heatmap
    cm = confusion_matrix(y_test, y_pred, labels=le.classes_)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, cmap="Blues", cbar=False,
                xticklabels=le.classes_,
                yticklabels=le.classes_)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    # Return results for barplot summary
    return {
        "Model": model_name,
        "Accuracy": acc,
        "MacroF1": macro_f1,
        "WeightedF1": weighted_f1,
        "NIR": nir,
        "TrainedModel": clf,   
        "LabelEncoder": le     
    }

In [ ]:
from sklearn.linear_model import LogisticRegression

results = []

# Primary only
clf = LogisticRegression(max_iter=500,
        solver='saga',      
        n_jobs=-1)
results.append(evaluate_model(clf, X_train_p, X_test_p, y_train, y_test, "Primary Only"))

# Mutations only
clf = LogisticRegression(max_iter=500,
        solver='saga',     
        n_jobs=-1)
results.append(evaluate_model(clf, X_train_m, X_test_m, y_train, y_test, "Mutations Only"))

# Combined
clf = LogisticRegression(max_iter=500,
        solver='saga',      # handles multinomial + sparse data
        n_jobs=-1)
results.append(evaluate_model(clf, X_train_c, X_test_c, y_train, y_test, "Combined"))

In [ ]:
df_results = pd.DataFrame(results)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"], 
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

## XGBoost

In [ ]:
from xgboost import XGBClassifier
results = []

res_primary = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_p, X_test_p, y_train, y_test,
    model_name="XGBoost - Primary Only"
)
results.append(res_primary)

res_mutations = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_m, X_test_m, y_train, y_test,
    model_name="XGBoost - Mutations Only"
)
results.append(res_mutations)

res_combined = evaluate_model(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42
    ),
    X_train_c, X_test_c, y_train, y_test,
    model_name="XGBoost - Combined"
)
results.append(res_combined)

In [ ]:
df_results = pd.DataFrame(results)

df_results.plot(x="Model", y=["Accuracy", "MacroF1", "WeightedF1"], 
                kind="bar", figsize=(8,6))
plt.title("Performance Comparison Across Feature Sets")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)  # metrics are between 0 and 1
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt
import os

model = res_combined["TrainedModel"]
le = res_combined["LabelEncoder"]

# Use the test set for SHAP
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_c)

# Global importance (bar plot)
plt.figure(figsize=(14, 10)) 
shap.summary_plot(shap_values, X_test_c, plot_type="bar", class_names=le.classes_,max_display=20, show=False)
ax = plt.gca()
for lbl in ax.get_yticklabels():
    lbl.set_fontsize(7)      # or 6–8
    lbl.set_va('center')

ax = plt.gca()

# Shrink x-axis tick labels
ax.tick_params(axis='x', labelsize=8)   # try 6–9

# Shrink x-axis title (the long SHAP caption)
ax.xaxis.label.set_size(8)              # try 8–10
ax.set_title("Global Feature Importance (SHAP)", fontsize=12, pad=10)

plt.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),   # move legend below chart
    ncol=5,                        # number of columns (adjust)
    fontsize="small",
    frameon=False
)
#plt.tight_layout()


# After all plotting calls
out_path = str(OUTPUT_DIR / "shap_importance.png")   # or .pdf/.svg for vector (best for print)
plt.savefig(out_path, dpi=300, bbox_inches='tight')   # 300–600 dpi for print

plt.show()

## Experimenting with TabTransformer

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ------------------------
# 1. Load data (replace with your files)
# ------------------------
clinical = pd.read_csv(str(CLINICAL_TSV), sep="\t")   # MSK clinical
clinical = clinical.dropna(subset=["Metastatic Site"])
mutations = pd.read_csv(str(MUTATIONS_MAF), sep="\t", comment='#', low_memory=False)


In [ ]:
# Keep only needed columns
clinical = clinical[["Sample ID", "Primary Tumor Site", "Metastatic Site"]]
mutations = mutations[["Hugo_Symbol", "Tumor_Sample_Barcode"]]

clinical = clinical.rename(columns={'Sample ID': 'Tumor_Sample_Barcode'})

trmerged = clinical.merge(mutations,
                  how='inner',
                  on='Tumor_Sample_Barcode')


mut_per_patient = (
    trmerged.groupby(["Tumor_Sample_Barcode", "Primary Tumor Site", "Metastatic Site"])["Hugo_Symbol"]
      .apply(lambda x: list(set(x)))  # unique mutations per patient
      .reset_index()
)

# Step 2: choose top-k mutation slots per patient (e.g., k=20)
k = 20
mut_per_patient["Mut_Slots"] = mut_per_patient["Hugo_Symbol"].apply(
    lambda muts: muts[:k] + ["None"]*(k - len(muts)) if len(muts) < k else muts[:k]
)

# Step 3: expand into columns MUT_1, MUT_2, ..., MUT_k
mut_expanded = pd.DataFrame(mut_per_patient["Mut_Slots"].to_list(), columns=[f"MUT_{i+1}" for i in range(k)])

# Step 4: final dataset
df_wide = pd.concat([mut_per_patient.drop(columns=["Hugo_Symbol", "Mut_Slots"]), mut_expanded], axis=1)

df_wide.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Define columns
categorical_cols = ["Primary Tumor Site"] + [f"MUT_{i+1}" for i in range(k)]
target_col = "Metastatic Site"

# -----------------------
# Step 1. Merge rare classes into "Other"
# -----------------------
min_count = 5  # threshold for rare classes
value_counts = df_wide[target_col].value_counts()

# Replace rare classes
df_wide[target_col] = df_wide[target_col].apply(
    lambda x: x if value_counts[x] >= min_count else "Other"
)

# -----------------------
# Step 2. Encode categorical + target columns
# -----------------------
encoders = {}
for col in categorical_cols + [target_col]:
    le = LabelEncoder()
    df_wide[col] = le.fit_transform(df_wide[col].astype(str))
    encoders[col] = le

# -----------------------
# Step 3. Build matrices
# -----------------------
X = df_wide[categorical_cols].values
y = df_wide[target_col].values

In [ ]:
from sklearn.model_selection import train_test_split
import torch

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.long)
X_test  = torch.tensor(X_test, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
import torch
import torch.nn as nn

# Number of categories per feature
categories = [len(encoders[col].classes_) for col in categorical_cols]
num_classes = len(encoders[target_col].classes_)

print("Categories per feature:", categories)
print("Number of classes:", num_classes)

class TabTransformer(nn.Module):
    def __init__(self, categories, num_classes, dim=32, depth=6, heads=8, mlp_hidden=64):
        super().__init__()
        # Embeddings per categorical feature
        self.embeddings = nn.ModuleList([
            nn.Embedding(cat_size, dim) for cat_size in categories
        ])
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=mlp_hidden, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        # Classifier head
        self.fc = nn.Sequential(
            nn.Linear(dim * len(categories), 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    def forward(self, x_categ):
        embs = [emb(x_categ[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.stack(embs, dim=1)      # (batch, n_features, dim)
        x = self.transformer(x)           # transformer output
        x = x.flatten(1)                  # flatten all features
        return self.fc(x)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TabTransformer(categories=categories, num_classes=num_classes).to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Move data to GPU if available
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test   = X_test.to(device), y_test.to(device)

epochs = 80
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    preds = outputs.argmax(dim=1)
    acc = accuracy_score(y_train.cpu(), preds.cpu())

    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | Train Acc: {acc:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    test_preds = test_outputs.argmax(dim=1)

test_acc = accuracy_score(y_test.cpu(), test_preds.cpu())
test_f1 = f1_score(y_test.cpu(), test_preds.cpu(), average="weighted")

print("\n=== Test Results ===")
print(f"Accuracy: {test_acc:.4f}")
print(f"Weighted F1: {test_f1:.4f}")

## Embedding

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# -----------------------
# Step 1. Extract features
# -----------------------
primary_site = df_wide["Primary Tumor Site"].values
mut_features = df_wide[[f"MUT_{i+1}" for i in range(k)]].values  # binary mutation features
y = df_wide["Metastatic Site"].values

# Train/test split
site_train, site_test, mut_train, mut_test, y_train, y_test = train_test_split(
    primary_site, mut_features, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to torch tensors
site_train = torch.tensor(site_train, dtype=torch.long)
mut_train  = torch.tensor(mut_train, dtype=torch.float32)
y_train    = torch.tensor(y_train, dtype=torch.long)

site_test = torch.tensor(site_test, dtype=torch.long)
mut_test  = torch.tensor(mut_test, dtype=torch.float32)
y_test    = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(site_train, mut_train, y_train)
test_ds  = TensorDataset(site_test, mut_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64)

In [ ]:
class OrganotropismNet(nn.Module):
    def __init__(self, n_sites, n_numeric, embedding_dim=8, hidden_dim=64, n_classes=5):
        super().__init__()
        
        # Embedding for primary tumor site
        self.site_emb = nn.Embedding(n_sites, embedding_dim)
        
        # Fully connected layers
        input_dim = embedding_dim + n_numeric
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim//2)
        self.out = nn.Linear(hidden_dim//2, n_classes)
    
    def forward(self, site, numeric):
        site_vec = self.site_emb(site)  # (batch, embedding_dim)
        x = torch.cat([site_vec, numeric], dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

In [ ]:
n_sites = len(encoders["Primary Tumor Site"].classes_)
n_numeric = mut_train.shape[1]
n_classes = len(encoders["Metastatic Site"].classes_)

model = OrganotropismNet(n_sites=n_sites, n_numeric=n_numeric, n_classes=n_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):
    model.train()
    total_loss = 0
    for site_batch, mut_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(site_batch, mut_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}")

In [ ]:
from sklearn.metrics import classification_report

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for site_batch, mut_batch, y_batch in test_loader:
        logits = model(site_batch, mut_batch)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(y_batch.numpy())
        y_pred.extend(preds.numpy())

print(classification_report(y_true, y_pred, target_names=encoders["Metastatic Site"].classes_))

## Binary prediction: primary tumour vs metastasis


In [ ]:
import pandas as pd
df=pd.read_csv(str(CLINICAL_TSV), sep="\t")


# === 1. Load the mutation file (MAF format) ===
# Adjust path to where you saved data_mutations.txt
mutations = pd.read_csv(str(MUTATIONS_MAF), sep="\t", comment='#', low_memory=False)

print("File shape:", mutations.shape)
print("Columns:", mutations.columns.tolist())
print(mutations.head())

# === 2. Inspect key fields ===
# Usually, 'Hugo_Symbol' = gene, 'Tumor_Sample_Barcode' = sample ID
print(mutations[['Hugo_Symbol', 'Tumor_Sample_Barcode', 'Variant_Classification']].head())

# === 3. Create a binary mutation matrix ===
# Pivot so each gene is a column, each sample is a row
# Mark 1 if mutated, 0 otherwise
binary_matrix = (
    mutations
    .assign(value=1)
    .pivot_table(index='Tumor_Sample_Barcode',
                 columns='Hugo_Symbol',
                 values='value',
                 aggfunc='max',
                 fill_value=0)
)

print("Binary matrix shape:", binary_matrix.shape)
print(binary_matrix.iloc[0:5, 0:10])  # show first 5 samples × first 10 genes

df = df.rename(columns={'Sample ID': 'Tumor_Sample_Barcode'})

merged = df.merge(binary_matrix,
                  how='inner',
                  on='Tumor_Sample_Barcode')

In [ ]:
target_col = "Sample Type"   # binary target
y = (merged[target_col] == "Metastasis").astype(int).values  # 1 = Metastasis, 0 = Primary

In [ ]:
# The clinical table has 27 columns; everything after them came from the
# mutation binary matrix, so it is the gene feature block.
gene_columns = list(merged.columns[27:])
categorical_cols = ["Primary Tumor Site", "Smoking History"] + gene_columns
numeric_cols = ["Mutation Count"]

X_categ = merged[categorical_cols].astype(str)
X_num   = merged[numeric_cols]

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    pd.concat([X_categ, X_num], axis=1), y, test_size=0.2, random_state=42, stratify=y
)

# Column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

In [ ]:
# Logistic Regression pipeline
log_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# Train
log_reg.fit(X_train, y_train)

# Predictions
y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 (weighted):", f1_score(y_test, y_pred, average="weighted"))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression (Metastasis Prediction)")
plt.legend(loc="lower right")
plt.show()

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1])  # balance classes
)

xgb.fit(preprocessor.fit_transform(X_train), y_train)
y_pred = xgb.predict(preprocessor.transform(X_test))
y_prob = xgb.predict_proba(preprocessor.transform(X_test))[:, 1]
# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 (weighted):", f1_score(y_test, y_pred, average="weighted"))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="darkgreen", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - XGBoost (Metastasis Prediction)")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Get feature names from the preprocessor
feature_names = preprocessor.get_feature_names_out()
import shap

# Use TreeExplainer for XGB
explainer = shap.TreeExplainer(xgb)

# Transform X_test through preprocessor
X_test_transformed = preprocessor.transform(X_test)

# Compute SHAP values
shap_values = explainer.shap_values(X_test_transformed)

# Summary plot (global importance)
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names)

In [ ]:
import numpy as np
import pandas as pd

# Mean absolute SHAP value per feature
mean_abs_shap = np.abs(shap_values).mean(axis=0)

# Put into dataframe
shap_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": mean_abs_shap
}).sort_values(by="importance", ascending=False)

topN = 20
print(shap_importance.head(topN))

import matplotlib.pyplot as plt

shap_importance.head(topN).plot(
    kind="barh", x="feature", y="importance", legend=False, figsize=(8,6)
)
plt.gca().invert_yaxis()
plt.title(f"Top {topN} Features by SHAP Importance (Mean Absolute)")
plt.xlabel("Mean |SHAP value|")
plt.show()